# 🎯 Resume-JD Gap Analyzer & Optimizer

**Pipeline:** Parse Resume → Gap Analysis → Writer↔Reviewer Loop → Generate PDF

**Models:** `qwen2.5:7b` (Writer) | `llama3.1:8b` (Reviewer)

---

## 1. Setup & Configuration

In [ ]:
import os, sys, json, yaml, time
from IPython.display import display, Markdown, HTML

# Load config
with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Setup LangSmith (optional — set your API key in config.yaml or env)
ls = config.get('langsmith', {})
if ls.get('enabled'):
    os.environ['LANGCHAIN_TRACING_V2'] = 'true'
    os.environ['LANGCHAIN_PROJECT'] = ls.get('project_name', 'resume-jd-optimizer')
    if ls.get('api_key'):
        os.environ['LANGCHAIN_API_KEY'] = ls['api_key']
    print('🔗 LangSmith tracing enabled')
else:
    os.environ['LANGCHAIN_TRACING_V2'] = 'false'
    print('ℹ️  LangSmith tracing disabled')

print(f"Writer model:   {config['llm']['writer_model']}")
print(f"Reviewer model: {config['llm']['reviewer_model']}")
print(f"Max iterations: {config['agent']['max_iterations']}")
print(f"Threshold:      {config['agent']['approval_threshold']}%")
print('✅ Config loaded')

## 2. Load & Parse Resume

In [ ]:
from utils.pdf_reader import extract_resume_text, extract_resume_sections

# Resume path (configured in config.yaml)
resume_path = config['pdf']['source_resume']
print(f'📄 Loading: {resume_path}')

# Extract raw text
resume_text = extract_resume_text(resume_path)
print(f'   Characters: {len(resume_text)}')

# Extract structured sections
sections = extract_resume_sections(resume_path)
print(f'   Sections: {len(sections)}')
print()

for name, bullets in sections.items():
    print(f'📌 {name} ({len(bullets)} items)')
    for b in bullets[:2]:
        print(f'   • {b[:100]}...' if len(b) > 100 else f'   • {b}')
    if len(bullets) > 2:
        print(f'   ... and {len(bullets)-2} more')

## 3. Enter Job Description

Paste the target job description below:

In [ ]:
# ✏️ PASTE YOUR JOB DESCRIPTION HERE
jd_text = """
Paste your job description here...
"""

print(f'📋 JD loaded: {len(jd_text)} characters')
print(f'   Preview: {jd_text.strip()[:200]}...')

## 4. Run Optimization Pipeline

This runs the full LangGraph pipeline: Gap Analysis → Writer↔Reviewer Loop → PDF Generation

In [ ]:
from pipeline import run_optimization

print('🚀 Starting optimization...')
print('=' * 60)

result = run_optimization(
    resume_path=resume_path,
    jd_text=jd_text.strip(),
    config=config,
)

print('\n' + '=' * 60)
print(f'✅ Done in {result["elapsed_seconds"]:.1f}s')
print(f'📊 Final Score: {result["final_score"]}%')
print(f'📄 PDF: {result["pdf_path"]}')
print(f'📋 Report: {result["report_path"]}')

## 5. Gap Analysis Report

In [ ]:
# Display report in notebook
display(Markdown(result['report_content']))

## 6. Iteration History

In [ ]:
# Score progression chart (text-based)
history = result.get('history', [])
gap = result.get('gap_analysis', {})

print('📈 Score Progression:')
print(f'   Initial: {gap.get("match_score", "?"):>3}%  {"█" * (gap.get("match_score", 0) // 5)}')
for h in history:
    bar = '█' * (h['score'] // 5)
    status = '✅' if h['approved'] else '🔄'
    print(f'   Iter {h["iteration"]}:  {h["score"]:>3}%  {bar} {status}')

print(f'\n   Final:   {result["final_score"]:>3}%')

## 7. Output Files

In [ ]:
import os

pdf_path = result.get('pdf_path', '')
report_path = result.get('report_path', '')

print('📦 Generated Files:')
if pdf_path and os.path.exists(pdf_path):
    size_kb = os.path.getsize(pdf_path) / 1024
    print(f'   📄 Optimized Resume: {pdf_path} ({size_kb:.1f} KB)')
else:
    print('   ⚠️  PDF not found')

if report_path and os.path.exists(report_path):
    size_kb = os.path.getsize(report_path) / 1024
    print(f'   📋 Gap Report:       {report_path} ({size_kb:.1f} KB)')
else:
    print('   ⚠️  Report not found')

print('\n✅ All done! Open the PDF to review your optimized resume.')